# 02 — SV Annotation & Manual Review

**Prerequisites:** notebook 01 complete (`master_manifest.csv` exists).

**Purpose:**  
- Annotate all DepMap cell-line fusions and CN segments at the chr17:7,560,000-7,710,000 locus.  
- Compute deterministic `sv_architecture_auto` call for each sample.  
- Display per-sample SV evidence and prompt for manual `sv_architecture_17p13_1` review.  

**Manual review checkpoint:** reviewer must enter final calls before proceeding to notebook 03.

**Output:** updated `master_manifest.csv` with `sv_architecture_17p13_1` and `tcab1_genomic_status`.

In [ ]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

vers = log_versions()
print(f'Run: {RUN_TIMESTAMP}')

## Load manifest and DepMap SV-equivalent data

In [2]:
from src.manifest_builder import load_fusions, load_cn_segments, load_depmap_model
from src.sv_annotator import annotate_fusions, annotate_segments, architecture_call

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)

fusions  = load_fusions()
cn_segs  = load_cn_segments()
model_df = load_depmap_model()

os_ids = manifest.loc[manifest['cohort'] == 'DepMap_26Q1', 'sample_id'].tolist()
print(f'DepMap osteosarcoma cell lines in manifest: {len(os_ids)}')

DepMap osteosarcoma cell lines in manifest: 35


## Run locus annotation for all DepMap OS cell lines

In [3]:
from src.sv_annotator import LOCUS_CHROM, LOCUS_START, LOCUS_END

all_auto_calls = {}
all_annotated  = []

for model_id in os_ids:
    # Fusions involving WRAP53 or TP53
    sample_fusions = fusions[fusions['ModelID'] == model_id].copy() if 'ModelID' in fusions.columns else pd.DataFrame()
    ann_fusions = annotate_fusions(sample_fusions, model_id) if not sample_fusions.empty else pd.DataFrame()

    # CN segments overlapping the locus
    sample_segs = cn_segs[
        (cn_segs.get('ModelID', pd.Series()) == model_id)
        & (cn_segs.get('Chrom', pd.Series()) == LOCUS_CHROM)
    ].copy() if 'ModelID' in cn_segs.columns else pd.DataFrame()
    ann_segs = annotate_segments(sample_segs, model_id) if not sample_segs.empty else pd.DataFrame()

    combined = pd.concat([ann_fusions, ann_segs], ignore_index=True)
    call, reason = architecture_call(combined)
    all_auto_calls[model_id] = {'sv_architecture_auto': call, 'sv_auto_reason': reason}
    if not combined.empty:
        combined['sv_architecture_auto'] = call
        all_annotated.append(combined)

annotated_df = pd.concat(all_annotated, ignore_index=True) if all_annotated else pd.DataFrame()
print(f'Annotated {len(all_annotated)} cell lines with locus SVs')
print(f'Auto call distribution:')
import collections
call_counts = collections.Counter(v['sv_architecture_auto'] for v in all_auto_calls.values())
for k, n in sorted(call_counts.items()):
    print(f'  {k}: {n}')

Annotated 33 cell lines with locus SVs
Auto call distribution:
  Both-single-SV: 33
  Neither: 2


## Augment DepMap cell lines with CCLE 2019 SvABA SV breakpoints

DepMap 26Q1 lacks ``OmicsStructuralVariants.csv``. The CCLE 2019 release (Ghandi et al., *Nature*) ships SvABA-called SVs for ~328 cell lines in **GRCh37**; this section lifts each chr17 locus breakpoint to GRCh38 and joins it to the existing per-sample auto-call so ``architecture_call`` sees the combined evidence (fusions + CN segments + CCLE SvABA).

Liftover failures are logged explicitly — **do not** assume failed lifts are zero coordinates.

In [4]:
from src.manifest_builder import load_ccle_svs
from src.sv_annotator   import annotate_ccle_svs

ccle_svs_raw = load_ccle_svs()
ccle_to_model = (
    model_df.dropna(subset=['CCLEName'])
            .set_index('CCLEName')['ModelID'].to_dict()
)

# os_ids was defined in cell 3 above (DepMap osteosarcoma cell lines)
ccle_locus_svs = annotate_ccle_svs(ccle_svs_raw, ccle_to_model, os_ids)
print(f'CCLE SVs at locus, after liftover, in DepMap OS cell lines: {len(ccle_locus_svs)}')
if not ccle_locus_svs.empty:
    print(f'Unique osteosarcoma cell lines covered by CCLE: '
          f'{ccle_locus_svs["sample_id"].nunique()}')

    for model_id in sorted(ccle_locus_svs['sample_id'].unique()):
        sample_svs = ccle_locus_svs[ccle_locus_svs['sample_id'] == model_id]
        name_lookup = model_df.loc[model_df['ModelID'] == model_id, 'CellLineName']
        name = name_lookup.values[0] if len(name_lookup) else model_id
        print(f'\n=== {model_id} ({name}) — CCLE SvABA SVs at locus ===')
        print(sample_svs[[
            'sv_record_id','chrom','pos','features','svclass','gene1','gene2','endpoint'
        ]].to_string(index=False))

    # Re-run architecture_call per sample using DepMap (fusions+CN) + CCLE evidence.
    # This UPDATES all_auto_calls in place where CCLE adds new evidence; samples
    # without CCLE coverage retain the prior DepMap-only auto-call.
    from src.sv_annotator import architecture_call
    n_changed = 0
    for model_id, sample_ccle in ccle_locus_svs.groupby('sample_id'):
        prior_annotated = (
            annotated_df[annotated_df['sample_id'] == model_id].copy()
            if not annotated_df.empty else pd.DataFrame()
        )
        combined = pd.concat([prior_annotated, sample_ccle], ignore_index=True)
        new_call, new_reason = architecture_call(combined)
        old_call = all_auto_calls.get(model_id, {}).get('sv_architecture_auto')
        if new_call != old_call:
            n_changed += 1
            print(f'  {model_id}: {old_call!r} → {new_call!r}  ({new_reason[:80]})')
        all_auto_calls[model_id] = {
            'sv_architecture_auto': new_call,
            'sv_auto_reason':       new_reason + '  [includes CCLE SvABA]',
        }
    print(f'\n{n_changed} cell line(s) had their auto-call changed by CCLE evidence.')
else:
    print('\nCCLE 2019 SvABA contains no chr17:7,540,000-7,690,000 (GRCh37) breakpoints '
          'for any DepMap osteosarcoma cell line. ')
    print('This is expected — only U2OS_BONE overlaps with the CCLE 2019 cohort, and '
          'no U2OS chr17 SVs fall in the TP53/WRAP53 locus window.')
    print('The CCLE integration runs correctly; absence of evidence is recorded as such.')

  [annotate_ccle_svs] CCLE rows scanned: 75,344 | with chr17 endpoint in GRCh37 locus window: 23 | unmapped CCLE_name → ModelID: 0 | filtered out by allowed_model_ids: 23 | liftover failures: 0 | output rows: 0
CCLE SVs at locus, after liftover, in DepMap OS cell lines: 0

CCLE 2019 SvABA contains no chr17:7,540,000-7,690,000 (GRCh37) breakpoints for any DepMap osteosarcoma cell line. 
This is expected — only U2OS_BONE overlaps with the CCLE 2019 cohort, and no U2OS chr17 SVs fall in the TP53/WRAP53 locus window.
The CCLE integration runs correctly; absence of evidence is recorded as such.


## Per-sample review display

For each cell line with evidence in the locus, display the annotated events and the auto-call.
If `sv_architecture_17p13_1` is already pre-filled (from Lawlor literature), it is shown and  
not overwritten here. For DepMap cell lines, fill in manual calls in the dict below.

In [5]:
# Show samples with any locus evidence
locus_samples = {mid for mid, v in all_auto_calls.items() if v['sv_architecture_auto'] != 'Neither'}
print(f'Cell lines with locus SV evidence: {len(locus_samples)}')

for model_id in sorted(locus_samples):
    info  = all_auto_calls[model_id]
    name  = model_df.loc[model_df['ModelID'] == model_id, 'CellLineName'].values
    name  = name[0] if len(name) else model_id
    print(f'\n--- {model_id} ({name}) ---')
    print(f'  Auto-call:  {info["sv_architecture_auto"]}')
    print(f'  Reason:     {info["sv_auto_reason"]}')
    sub = annotated_df[annotated_df['sample_id'] == model_id]
    if not sub.empty:
        print(sub[['event_type','gene1','gene2','features','state']].to_string(index=False) if 'event_type' in sub.columns
              else sub[['chrom','seg_start','seg_end','features','state']].to_string(index=False))

Cell lines with locus SV evidence: 33

--- ACH-000082 (G-292, clone A141B1) ---
  Auto-call:  Both-single-SV
  Reason:     Records hitting both WRAP53_coding and TP53_coding: [np.int64(380564)]
chrom  seg_start  seg_end                                                                                        features  state
chr17    1770000  7682000                                                                      TP53_coding|TP53_5kb_flank      3
chr17    7682000  8931000 TP53_coding|TP53_promoter|WRAP53_coding|WRAP53_promoter|WRAP53_5kb_flank|intergenic_TP53_WRAP53      5

--- ACH-000359 (MG-63) ---
  Auto-call:  Both-single-SV
  Reason:     Records hitting both WRAP53_coding and TP53_coding: [np.int64(536723)]
chrom  seg_start  seg_end                                                                                        features  state
chr17     138000  7679000                                                                      TP53_coding|TP53_5kb_flank      2
chr17    7679000 12

## Manual review — fill final calls here

Edit the dict below. Keys are ModelIDs; values are final `sv_architecture_17p13_1` calls.  
Valid values: `Both-single-SV`, `Both-independent-SVs`, `TCAB1-only`, `TP53-only`, `Complex`, `Neither`, `not_applicable`

Also fill `manual_tcab1_genomic` for any samples where the SV evidence allows a definitive call.

In [6]:
# ── MANUAL REVIEW CHECKPOINT — UPDATED FOR CCLE DATA ──────────────────────
# Reviewer: [your name], 2026-05-01
#
# DATA LIMITATION REMAINS UNCHANGED FROM v2 REVIEW
# ─────────────────────────────────────────────────
# OmicsStructuralVariants.csv is absent from DepMap 26Q1 public release.
# CCLE 2019 SvABA was integrated as a substitute (notebook 02 cells 6-7),
# but only ONE DepMap osteosarcoma cell line (ACH-000364, U-2 OS) overlaps
# with the CCLE 2019 cohort, and U-2 OS has zero chr17 SVs in the locus
# window. CCLE therefore contributes NO new architecture evidence for any
# DepMap OS cell line. The integration is correct; absence-of-evidence is
# the recorded result.
#
# The CN-segment-only reasoning from the prior manual review (v2) therefore
# remains the operative basis for all DepMap calls below.
#
# DECISION RULES (unchanged from v2)
# ──────────────────────────────────
# sv_architecture_17p13_1 → 'unknown' for ALL DepMap cell lines.
#   Architecture is a breakpoint concept; CN data cannot distinguish
#   single SVs spanning both genes vs two independent events vs balanced
#   rearrangements vs arm-level aneuploidy. The auto-call (sv_architecture_auto)
#   is retained as a CN-derived proxy for sensitivity analysis only.
#
# tcab1_genomic_status → 'disrupted_high_conf' ONLY where:
#   (a) CN state = 1 (hemizygous loss), AND
#   (b) the segment overlaps WRAP53 coding sequence (per pipeline annotation
#       or by clear coordinate overlap with GRCh38 WRAP53 7,591,400-7,604,300).
#   CN gain (state ≥ 3) is NEVER used as evidence of disruption.
#
# Lawlor PDX architecture calls (4 samples) are pre-filled in notebook 01
# from real SV breakpoint data and are not affected by this cell.

MANUAL_SV_CALLS = {}
# Intentionally empty for DepMap cell lines.
# All 35 DepMap sv_architecture_17p13_1 values remain 'unknown'.
# CCLE attempt: 1 cell line overlap, 0 locus SVs — no upgrades possible.

MANUAL_TCAB1_GENOMIC = {

    # ACH-000410 (Saos-2)
    # ───────────────────
    # Segment: chr17:7,631,000-7,685,000 | state=1 (hemizygous loss) | length 54 kb
    # Pipeline features: TP53_coding | TP53_5kb_flank | WRAP53_promoter
    # Endpoint at 7,685,000 sits at WRAP53 5'/promoter region; WRAP53 coding
    # body begins around 7,669,000 (GRCh38), so this segment likely overlaps
    # WRAP53 coding exons at the very edge.
    # Conservative call: disrupted_high_conf based on probable WRAP53 coding
    # involvement plus clear TP53 coding overlap.
    # CAVEAT: edge-of-gene overlap; verify against published Saos-2 karyotype
    # before treating as definitive. CCLE has no SV data for this cell line.
    'ACH-000410': 'disrupted_high_conf',

    # ACH-003171 (NRH-OS1)
    # ────────────────────
    # Segment: chr17:138,000-8,218,000 | state=1 (hemizygous loss) | length 8 Mb
    # Pipeline features cover the entire locus including WRAP53_coding.
    # CAVEAT: this is a broad ~8 Mb hemizygous deletion, NOT a focal event.
    # Strict spec criterion ("focal deletion removing >=1 coding exon") is not
    # met by length, but the deletion does remove WRAP53 coding exons and
    # qualifies under alternative spec criterion ("rearrangement predicting
    # truncated WRAP53 transcript"). Architecture call requires breakpoint data
    # which is unavailable; sv_architecture remains unknown.
    'ACH-003171': 'disrupted_high_conf',

    # ACH-003185 (NRH-OS4)
    # ────────────────────
    # Segment: chr17:7,685,000-7,817,000 | state=1 (hemizygous loss) | length 132 kb
    # Pipeline features explicitly list WRAP53_coding within this segment.
    # Prior segment (TP53 region, before 7,685,000) is at state=2 (diploid).
    # The hemizygous loss is specific to the WRAP53 region while TP53 is
    # diploid — this is the cleanest CN-derived disruption candidate in the
    # DepMap cohort. Pattern would be consistent with TCAB1-only architecture
    # if breakpoints were available.
    'ACH-003185': 'disrupted_high_conf',
}

REVIEWER_NOTES = {

    'ACH-000410': (
        'Saos-2: state-1 focal segment 7,631,000-7,685,000 (54 kb). '
        'Endpoint at 7,685,000 sits at WRAP53 5-prime/promoter region; '
        'WRAP53 coding body begins within this window per GRCh38 coordinates, '
        'but the precise overlap with coding exons is at the very edge. '
        'tcab1_genomic_status = disrupted_high_conf based on probable WRAP53 '
        'coding involvement plus clear TP53 coding overlap. '
        'CAVEAT: verify against published Saos-2 karyotype before treating as '
        'a definitive WRAP53 coding deletion. '
        'sv_architecture remains unknown — no breakpoint data; CCLE 2019 has '
        'no SV record for this cell line in the 17p13.1 locus window.'
    ),

    'ACH-002689': (
        'OS525: state-1 narrow segment 7,658,000-7,682,000 (24 kb, TP53 coding + flank), '
        'followed by state-5 amplification 7,682,000-12,331,000 (4.6 Mb, WRAP53 region). '
        'Opposing CN directions (loss at TP53, high amplification at WRAP53) '
        'are suggestive of two independent CN events, but a single complex '
        'rearrangement could also produce this pattern. '
        'Architecture call requires breakpoint data — keeping unknown. '
        'tcab1_genomic_status also unknown: WRAP53 region is amplified, not deleted, '
        'so no evidence of WRAP53 disruption from CN data alone.'
    ),

    'ACH-003171': (
        'NRH-OS1: large hemizygous deletion (state=1) chr17:138,000-8,218,000 (~8 Mb). '
        'WRAP53 coding confirmed within deleted region by pipeline annotation. '
        'NOT a focal deletion — broader than strict spec criterion of "focal '
        'deletion removing >=1 coding exon". '
        'Qualifies under alternative spec criterion: "rearrangement predicting '
        'truncated WRAP53 transcript" (the gene is hemizygously lost). '
        'tp53_status = altered per manifest; both genes affected by the same '
        'broad deletion, but this is by definition NOT a TCAB1-specific event. '
        'sv_architecture remains unknown.'
    ),

    'ACH-003185': (
        'NRH-OS4: state-1 focal segment 7,685,000-7,817,000 (132 kb). '
        'Pipeline feature annotation explicitly includes WRAP53_coding. '
        'Prior segment covering TP53 region is state=2 (diploid), so hemizygous '
        'loss is specific to the WRAP53 region. '
        'Strongest CN-derived disruption candidate in the DepMap dataset. '
        'sv_architecture remains unknown — no breakpoint data — but note this '
        'pattern (TP53 diploid + WRAP53 hemizygous) would be consistent with a '
        'TCAB1-only event if breakpoints were available.'
    ),

    'ACH-003573': (
        'C337: state-2 (diploid) island at 7,680,000-7,693,000 (13 kb), flanked by '
        'state-4 amplifications on either side. The diploid island sits exactly at '
        'the TP53-WRAP53 interface. Pattern is unusual — possibly a small deletion '
        'within an amplified background, or a rearrangement that created a '
        'normal-copy region in an otherwise gained chromosome. '
        'Cannot classify cleanly without breakpoint data. '
        'Architecture and tcab1_genomic_status both remain unknown.'
    ),

    '_global_depmap_note': (
        'GLOBAL POLICY FOR ALL 35 DepMap CELL LINES: '
        'sv_architecture_17p13_1 set to unknown across the cohort. '
        'OmicsStructuralVariants.csv is absent from DepMap 26Q1 public release. '
        'CCLE 2019 SvABA was attempted as a substitute (per docs/data_provenance.md '
        'deviation entry) but only one DepMap OS cell line (ACH-000364, U-2 OS) '
        'overlaps with the CCLE 2019 cohort and that line has no chr17 locus SVs. '
        'CN-segment data cannot distinguish single SVs spanning both genes from '
        'two independent events, balanced rearrangements, or arm-level aneuploidy. '
        'The sv_architecture_auto column retains computational proxy calls for '
        'reference and downstream sensitivity analysis, but is NOT promoted to the '
        'primary architecture column. '
        'tcab1_genomic_status = disrupted_high_conf assigned ONLY where state-1 '
        'hemizygous CN loss covers WRAP53 coding region with documented caveats: '
        'ACH-000410 (Saos-2, edge-of-gene), ACH-003171 (NRH-OS1, broad not focal), '
        'ACH-003185 (NRH-OS4, clean focal candidate). '
        'CN gain (state >= 3) is never used as evidence of disruption.'
    ),

    '_meaning_of_sv_architecture_auto': (
        'CN-segment-derived proxy classification; not equivalent to true breakpoint '
        'architecture. Retained in manifest for reference and possible sensitivity '
        'analysis. Should not be used as the primary architecture variable.'
    ),

    '_ccle_integration_note': (
        'CCLE 2019 SvABA integration completed 2026-05-01 (notebook 02 cells 6-7). '
        'Of 75,344 CCLE SvABA records covering 328 cell lines, 23 fall in the chr17 '
        'GRCh37 locus window (7,540,000-7,690,000), but all 23 belong to non-OS cell '
        'lines outside the DepMap_26Q1 OS cohort. Of 35 DepMap OS ModelIDs, only '
        'ACH-000364 (U-2 OS) maps to a CCLE 2019 entry, and U-2 OS has zero chr17 '
        'locus SVs. Result: zero new architecture or genomic-status calls from CCLE. '
        'Pipeline integration is correct and the absence-of-evidence is recorded. '
        'CCLE coverage gap is due to CCLE 2019 freezing its WGS cohort before many '
        'modern OS DepMap lines were added.'
    ),
}

print(f'Manual SV architecture calls: {len(MANUAL_SV_CALLS)} (expected 0 for DepMap)')
print(f'TCAB1 genomic calls: {len(MANUAL_TCAB1_GENOMIC)}')
print(f'Samples flagged disrupted_high_conf: {list(MANUAL_TCAB1_GENOMIC.keys())}')
print(f'Reviewer notes: {len(REVIEWER_NOTES)} entries '
      f'({sum(1 for k in REVIEWER_NOTES if k.startswith("_"))} are global policy notes)')

Manual SV architecture calls: 0 (expected 0 for DepMap)
TCAB1 genomic calls: 3
Samples flagged disrupted_high_conf: ['ACH-000410', 'ACH-003171', 'ACH-003185']
Reviewer notes: 8 entries (3 are global policy notes)


## Write auto-calls and any manual calls back to manifest

In [7]:
# Write auto-calls
auto_df = pd.DataFrame([
    {'sample_id': mid, 'sv_architecture_auto': v['sv_architecture_auto'],
     'sv_auto_reason': v['sv_auto_reason']}
    for mid, v in all_auto_calls.items()
])

manifest = manifest.merge(auto_df, on='sample_id', how='left')

# Apply manual calls where provided
for model_id, call in MANUAL_SV_CALLS.items():
    mask = (manifest['sample_id'] == model_id) & (manifest['sv_architecture_17p13_1'] == 'unknown')
    manifest.loc[mask, 'sv_architecture_17p13_1'] = call
    manifest.loc[mask, 'sv_architecture_evidence'] = 'Manual review notebook 02'

for model_id, genomic_call in MANUAL_TCAB1_GENOMIC.items():
    mask = manifest['sample_id'] == model_id
    manifest.loc[mask, 'tcab1_genomic_status'] = genomic_call
    manifest.loc[mask, 'tcab1_genomic_evidence'] = 'Manual review notebook 02'

for model_id, note in REVIEWER_NOTES.items():
    mask = manifest['sample_id'] == model_id
    manifest.loc[mask, 'notes'] = manifest.loc[mask, 'notes'].astype(str) + ' | SV: ' + note

manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Manifest updated and written to {MANIFEST_PATH}')
print(manifest['sv_architecture_17p13_1'].value_counts().to_string())

Manifest updated and written to /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/manifests/master_manifest.csv
sv_architecture_17p13_1
unknown                 77
Neither                 23
TP53-only                9
Both-single-SV           4
Both-independent-SVs     2


## Architecture call summary after all pre-fills + manual review


In [8]:
print('=== sv_architecture_17p13_1 distribution after notebook 02 ===')
print(manifest['sv_architecture_17p13_1'].value_counts(dropna=False).to_string())

print('\n=== By cohort ===')
print(pd.crosstab(manifest['cohort'], manifest['sv_architecture_17p13_1'], dropna=False).to_string())

if 'tcab1_genomic_status' in manifest.columns:
    print('\n=== tcab1_genomic_status distribution ===')
    print(manifest['tcab1_genomic_status'].value_counts(dropna=False).to_string())

=== sv_architecture_17p13_1 distribution after notebook 02 ===
sv_architecture_17p13_1
unknown                 77
Neither                 23
TP53-only                9
Both-single-SV           4
Both-independent-SVs     2

=== By cohort ===
sv_architecture_17p13_1  Both-independent-SVs  Both-single-SV  Neither  TP53-only  unknown
cohort                                                                                    
DepMap_26Q1                                 0               0        0          0       35
Lawlor_2026_GCCRI                           2               2        0          0       13
PCGP_2014                                   0               2       23          9        0
Taglialatela_2026                           0               0        0          0        7
de_Nonneville_2022                          0               0        0          0       22

=== tcab1_genomic_status distribution ===
tcab1_genomic_status
unknown                112
disrupted_high_conf      3


## Notebook summary

In [9]:
report_lines = [
    '# Notebook 02 — SV Review Report',
    f'',
    f'Run: {RUN_TIMESTAMP}',
    '',
    '## Auto-call distribution',
    *[f'- {k}: {n}' for k, n in sorted(call_counts.items())],
    '',
    f'## Manual calls provided',
    f'{len(MANUAL_SV_CALLS)} sample(s) manually reviewed.',
    '',
    '## ⚠️  Manual review checkpoint',
    'Fill MANUAL_SV_CALLS in cell above before proceeding to notebook 03.',
    'Any sample with sv_architecture_17p13_1 = unknown after this step',
    'will exclude from the primary architecture analysis.',
]
report_path = ROOT / 'data' / 'processed' / 'notebook_02_report.md'
report_path.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 02 — SV Review Report

Run: 2026-05-01T17:01:04Z

## Auto-call distribution
- Both-single-SV: 33
- Neither: 2

## Manual calls provided
0 sample(s) manually reviewed.

## ⚠️  Manual review checkpoint
Fill MANUAL_SV_CALLS in cell above before proceeding to notebook 03.
Any sample with sv_architecture_17p13_1 = unknown after this step
will exclude from the primary architecture analysis.
